## Lab 2: Visualizing Text

* Dataset: books.txt (Harry Potter series)
* Tasks (use the first 3 books; for each book):
  * Check whether all chapters in the book are of comparable length.
  * If they differ, identify which chapter is the shortest and which is the longest.
  * Count how often each of these main characters appears. As shown in the class, we’ll focus on these characters:
    * Harry Potter (Harry)
    * Ron Weasley (Ron)
    * Hermione Granger (Hermione)
    * Dobby
    * Draco Malfoy (Malfoy)
    * Lord Voldemort (Voldemort, You know who)
    * Professor Severus Snape (Snape)
    * Professor Albus Dumbledore (Dumbledore)
    * Rubeus Hagrid (Hagrid)
    * Sirius Black (Sirius)
  * Pick two proper visualizations to display character frequencies. Also provide a reason for choosing them.
  * Find out if Harry dominates every chapter, or are there shifts?

In [1]:
import re
import nltk
from nltk.tokenize import word_tokenize

In [2]:
from text_analytics.utils import extract_books
# 1. Load books directly (defaults to data/books.txt)
books = extract_books()

# 2. Access a specific book
target_book = "Harry Potter and the Sorcerer’s Stone"
sorcerer_stone = books[target_book]

print(f"Book title: {target_book}")
print(f"Total lines: {len(sorcerer_stone['content'])}")

Book title: Harry Potter and the Sorcerer’s Stone
Total lines: 3132


In [3]:
def get_chapters(book):
    chapters = []
    chapter_content = None
    for line in book['content']:
        if re.match(r'Chapter \d+', line):
            if chapter_content:
                chapters.append("\n".join(chapter_content))
            chapter_content = []
        else:
            if chapter_content is not None:
                chapter_content.append(line.lower())

    if chapter_content:
        chapters.append("\n".join(chapter_content))

    return chapters

In [4]:
sorcerer_chapters = get_chapters(sorcerer_stone)
sorcerer_chapters[0]

'the boy who lived\nmr. and mrs. dursley, of number four, privet drive, were proud to say that they were perfectly normal, thank you very much. they were the last people you’d expect to be in\xadvolved in anything strange or mysterious, because they just didn’t hold with such nonsense.\nmr. dursley was the director of a firm called grunnings, which made drills. he was a big, beefy man with hardly any neck, al\xadthough he did have a very large mustache. mrs. dursley was thin and blonde and had nearly twice the usual amount of neck, which came in very useful as she spent so much of her time craning over garden fences, spying on the neighbors. the dursleys had a small son called dudley and in their opinion there was no finer boy any\xadwhere.\nthe dursleys had everything they wanted, but they also had a secret, and their greatest fear was that somebody would discover it. they didn’t think they could bear it if anyone found out about the potters. mrs. potter was mrs. dursley’s sister, but

In [5]:
books.keys()

dict_keys(['Harry Potter and the Chamber of Secrets', 'Harry Potter and the Deathly Hallows', 'Harry Potter and the Goblet Of Fire', 'Harry Potter and the Half-Blood Prince', 'Harry Potter and the Order of the Phoenix', 'Harry Potter Great Hall', 'Harry Potter and the Prisoner of Azkaban', 'Harry Potter and the Sorcerer’s Stone'])

In [6]:
first_3_books = ["Harry Potter and the Sorcerer’s Stone","Harry Potter and the Chamber of Secrets","Harry Potter and the Prisoner of Azkaban"]

In [7]:
test_chapter = sorcerer_chapters[0]
test_chapter[:20]

'the boy who lived\nmr'

In [8]:
def get_chapter_length(chapter):
    chapter_token = word_tokenize(chapter)
    return chapter_token,len(chapter_token)

In [9]:
chapter_token,length = get_chapter_length(test_chapter)

In [10]:
chapter_token[:10],length

(['the', 'boy', 'who', 'lived', 'mr.', 'and', 'mrs.', 'dursley', ',', 'of'],
 5856)

In [11]:
def get_book_chapters_length(chapters):
    for chapter in chapters:
        token,length = get_chapter_length(chapter)
        yield {"token" : token, "length" : length}

In [12]:
book_token_length = list(get_book_chapters_length(sorcerer_chapters))

In [13]:
for item in book_token_length:
    print(item["token"][:5])
    print(item["length"])
    print()

['the', 'boy', 'who', 'lived', 'mr.']
5856

['the', 'vanishing', 'glass', 'nearly', 'ten']
4307

['the', 'letters', 'from', 'no', 'one']
4836

['the', 'keeper', 'of', 'the', 'keys']
4979

['diagon', 'alley', 'harry', 'woke', 'early']
8718

['the', 'journey', 'from', 'platform', 'nine']
8462

['the', 'sorting', 'hat', 'the', 'door']
5668

['the', 'potions', 'master', '“', 'there']
3738

['the', 'midnight', 'duel', 'harry', 'had']
6530

['halloween', 'malfoy', 'couldn', '’', 't']
5496

['quidditch', 'as', 'they', 'entered', 'november']
4359

['the', 'mirror', 'of', 'erised', 'christmas']
7049

['nicholas', 'flamel', 'dumbledore', 'had', 'convinced']
4219

['norbert', 'the', 'norwegian', 'ridgeback', 'quirrell']
4679

['the', 'forbidden', 'forest', 'things', 'couldn']
6856

['through', 'the', 'trapdoor', 'in', 'years']
8783

['the', 'man', 'with', 'two', 'faces']
7306



In [36]:
def count_character_in_chapter(chapter, character):
    # If character is a list of alias strings, run for each alias and sum the counts
    if isinstance(character, list):
        return sum(count_character_in_chapter(chapter, alias) for alias in character)

    # Base case: character is a single string
    char_token = word_tokenize(character.lower())

    if isinstance(chapter, list):
        chap_token = chapter
        chap_length = len(chap_token)
    else:
        chap_token, chap_length = get_chapter_length(chapter)

    chap_token = [t.lower() for t in chap_token]

    char_length = len(char_token)
    if char_length == 0 or chap_length < char_length:
        return 0

    char_count = 0
    for i in range(chap_length - char_length + 1):
        if chap_token[i : i + char_length] == char_token:
            char_count += 1

    return char_count

In [37]:
voldemort_aliases = ["Lord Voldemort", "Tom Marvolo", "You-Know-Who"]
test_text = "Lord Voldemort and Tom Marvolo Riddle were the same person. You-Know-Who returned."

# Pass the list of aliases directly:
total_count = count_character_in_chapter(test_text, voldemort_aliases)
print("Total count:", total_count)

Total count: 3


In [38]:
characters = {
    "harry" : ["Harry"],
    "ron" : ["Ron"],
    "hermione" : ["Hermione"],
    "dobby" : ["Dobby"],
    "malfoy" : ["Malfoy"],
    "voldermort" : ["Voldermort","Dark Lord", "You-Know-Who", "He Who Must Not Be Named", "Tom Marvolo Riddle"],
    "snape" : ["Snape"],
    "dumbledore" : ["Dumbledore"],
    "hagrid" : ["Hagrid"],
    "sirius" : ["Sirius"]
}

In [39]:
print(len(sorcerer_chapters))

17


In [40]:
import pandas as pd
from nltk.tokenize import word_tokenize

# Formula: count(First) + count(Last) - count(First Last)
def count_character_inc_exc(chapter, first_name, last_name):
    c_first = count_character_in_chapter(chapter, first_name)
    c_last = count_character_in_chapter(chapter, last_name)
    c_full = count_character_in_chapter(chapter, f"{first_name} {last_name}")
    return c_first + c_last - c_full

# Composer Function
def compose_character_df(chapters):
    voldemort_aliases = [
        "Voldemort", "Dark Lord", "You-Know-Who",
        "He Who Must Not Be Named", "Tom Marvolo Riddle"
    ]

    rows = []
    for idx, chapter in enumerate(chapters, 1):
        # Tokenize chapter once per iteration for maximum speed
        chap_tokens, length = get_chapter_length(chapter)

        row = {
            "chapter": f"Chapter {idx}",
            "chapter_length": length,
            "Harry Potter": count_character_inc_exc(chap_tokens, "harry", "potter"),
            "Ron Weasley": count_character_inc_exc(chap_tokens, "ron", "weasley"),
            "Hermione Granger": count_character_inc_exc(chap_tokens, "hermione", "granger"),
            "Dobby": count_character_in_chapter(chap_tokens, "dobby"),
            "Draco Malfoy": count_character_inc_exc(chap_tokens, "draco", "malfoy"),
            "Lord Voldemort": count_character_in_chapter(chap_tokens, voldemort_aliases),
            "Severus Snape": count_character_inc_exc(chap_tokens, "severus", "snape"),
            "Albus Dumbledore": count_character_inc_exc(chap_tokens, "albus", "dumbledore"),
            "Rubeus Hagrid": count_character_inc_exc(chap_tokens, "rubeus", "hagrid"),
            "Sirius Black": count_character_inc_exc(chap_tokens, "sirius", "black")
        }
        rows.append(row)

    return pd.DataFrame(rows)

In [41]:
df = compose_character_df(sorcerer_chapters)

In [42]:
df

,chapter,chapter_length,Harry Potter,Ron Weasley,Hermione Granger,Dobby,Draco Malfoy,Lord Voldemort,Severus Snape,Albus Dumbledore,Rubeus Hagrid,Sirius Black
0,Chapter 1,5856,24,0,0,0,0,10,0,36,14,3
1,Chapter 2,4307,79,0,0,0,0,0,0,0,0,1
2,Chapter 3,4836,72,0,0,0,0,0,0,0,0,1
3,Chapter 4,4979,53,0,0,0,0,5,0,13,38,3
4,Chapter 5,8718,156,0,0,0,0,2,0,3,97,5
5,Chapter 6,8462,119,72,8,0,8,9,0,10,13,4
6,Chapter 7,5668,67,19,6,0,5,0,5,11,4,5
7,Chapter 8,3738,56,25,10,0,4,0,21,0,27,3
8,Chapter 9,6530,78,41,19,0,37,0,1,2,5,1
9,Chapter 10,5496,85,48,31,0,9,0,6,3,0,3
